# 📓 Caderno de Exercícios — Aula 05: Visualização de Dados e Storytelling
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

> 💡 **Liberdade Criativa:** Na visualização de dados, você pode usar `seaborn`, `matplotlib` ou até `plotly`! O objetivo não é apenas plotar o gráfico, mas deixá-lo **limpo, com título claro, eixos nomeados e fácil de um diretor bater o olho e tomar uma decisão**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from difflib import get_close_matches

# Configurando o tema visual
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)

# === Checkpoint Inicial (Base 100% Limpa das Aulas 01 a 03) ===
df = pd.read_excel('Campe_Supply_Corrompido.xlsx', sheet_name='Campe Supply')
df = df.drop(columns=['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('ã', 'a')
df = df.drop_duplicates().reset_index(drop=True)

categorias_oficiais = {
    'setor': ['Congelados', 'Latícinio', 'Bebidas', 'Carnes'],
    'produto': ['Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada'],
    'vendedor': ['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro'],
    'regiao': ['Sudeste', 'Nordeste', 'Sul'],
    'estado': ['Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina'],
    'cliente': ['Atacarejo', 'Varejo', 'Atacado']
}
def corrigir_palavra(valor, lista_oficial):
    if pd.isna(valor): return np.nan
    match = get_close_matches(str(valor).strip(), lista_oficial, n=1, cutoff=0.6)
    return match[0] if match else np.nan

for col, lista_valida in categorias_oficiais.items():
    df[col] = df[col].apply(lambda x: corrigir_palavra(x, lista_valida))

mapa_produto_setor = {'Queijo': 'Latícinio', 'Leite': 'Latícinio', 'Coalhada': 'Latícinio', 'Sorvete': 'Congelados', 'Lasanha': 'Congelados', 'Pizza': 'Congelados', 'Frango': 'Carnes', 'Bovina': 'Carnes', 'Suína': 'Carnes', 'Refrigerantes': 'Bebidas', 'Cervejas': 'Bebidas', 'Sucos': 'Bebidas'}
mapa_estado_regiao = {'Rio de Janeiro': 'Sudeste', 'Minas Gerais': 'Sudeste', 'São Paulo': 'Sudeste', 'Bahia': 'Nordeste', 'Rio Grande do Norte': 'Nordeste', 'Sergipe': 'Nordeste', 'Rio Grande do Sul': 'Sul', 'Paraná': 'Sul', 'Santa Catarina': 'Sul'}
df['setor'] = df['setor'].fillna(df['produto'].map(mapa_produto_setor))
df['regiao'] = df['regiao'].fillna(df['estado'].map(mapa_estado_regiao))

for col in ['faturamento', 'margem_de_lucro', 'lucro']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.loc[df['lucro'] > df['faturamento'], 'lucro'] = df['faturamento'] * df['margem_de_lucro']
df.loc[(df['margem_de_lucro'] == 0) & (df['lucro'] > 0), 'margem_de_lucro'] = df['lucro'] / df['faturamento']
df.loc[(df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs() > 0.01, 'faturamento'] = df['lucro'] / df['margem_de_lucro']

df['lucro'] = df['lucro'].fillna(df['faturamento'] * df['margem_de_lucro']).round(2)
df['faturamento'] = df['faturamento'].fillna(df['lucro'] / df['margem_de_lucro']).round(2)
df['margem_de_lucro'] = df['margem_de_lucro'].fillna(df['lucro'] / df['faturamento']).round(2)
df = df.drop_duplicates().reset_index(drop=True)
df['mes'] = df['data_da_venda'].dt.month

## 🎯 Missão 1: Gráfico de Linha — Sazonalidade Mensal do Faturamento
Agrupe a soma de `faturamento` por `mes` (de 1 a 12) e plote um gráfico de linha (`sns.lineplot` ou `plt.plot`).

* **Dica Visual:** Coloque marcadores nos pontos (`marker='o'`) e renomeie o eixo X com os nomes dos meses (`['Jan', 'Fev', ..., 'Dez']`) usando `plt.xticks()`.
* 🧠 **Insight para descobrir no gráfico:** O que acontece com o faturamento da Campe Supply nos meses de **Novembro e Dezembro**?

In [ ]:
# 1. Plote o gráfico de linha da evolução mensal do faturamento
# Seu código aqui:


## 🎯 Missão 2: Gráfico de Barras Horizontais — Ranking de Vendedores
Agrupe o faturamento (ou lucro) total por `vendedor`, ordene do maior para o menor e crie um gráfico de barras horizontais (`sns.barplot(x=..., y=...)`).

* **Desafio Bônus:** Tente colocar os valores em `R$` escritos na ponta de cada barra usando `ax.bar_label()`!

In [ ]:
# 2. Plote o ranking de vendedores em barras horizontais
# Seu código aqui:


## 🎯 Missão 3: Boxplot — Distribuição do Ticket por Setor
Plote um Boxplot (`sns.boxplot`) cruzando `setor` no eixo X e `faturamento` no eixo Y.

* 🧠 **Reflexão:** Observe os pontos acima do limite superior (*outliers* reais de vendas altas no fim do ano). O que teria acontecido com esse gráfico se não tivéssemos limpado aquele erro de digitação de R$ 8.581,60 na Aula 03?

In [ ]:
# 3. Plote o Boxplot de faturamento por setor
# Seu código aqui:


## 🎯 Missão 4: Mapa de Calor (Heatmap) — Estado vs. Canal de Cliente
Crie uma `pivot_table` com a soma de `faturamento` tendo `estado` nas linhas e `cliente` nas colunas, e plote um Mapa de Calor com `sns.heatmap(..., annot=True, fmt='.0f', cmap='YlGnBu')`.

In [ ]:
# 4. Plote o Heatmap de Estado x Canal de Cliente
# Seu código aqui:
